In [1]:
!pip install pytorch_lightning -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 73.9 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
LitePep-Diff (end-to-end, TG-CDDPM-style)
- Stage-1: SciBERT (frozen) -> CLIP head (trainable 768->256) -> DDPM-Adapter(z_txt_clip, t) -> c_t
- Stage-2: PepEncoder (trainable) -> clean p0 [B,L,d0]
          CLIP uses p0 (AA-only pooled).
          PepProj MLP (trainable) -> p0_proj [B,L,d] for diffusion + per-t alignment.
- Stage-3: Diffusion creates p_t from p0_proj and ε at t, then channel-concat [p_t | broadcast(c_t)]
          -> fuse -> causal DiT -> eps_pred, dec_logits.
- Losses: L_diff (ε-MSE), L_ce (AA+EOS only), L_align_t (MSE(c_t, mean_AA(p_t))), L_clip (InfoNCE on p0 vs z_txt_clip).
"""

import os, sys, csv, json, math, time, random
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ========= Vocab =========
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"   # 20 AAs
IDX_EOS, IDX_PAD, IDX_BOS = 20, 21, 22
VOCAB_SIZE = 23
AA2IDX = {aa: i for i, aa in enumerate(AA_ALPHABET)}
IDX2AA = {i: aa for aa, i in AA2IDX.items()}

def encode_seq(seq: str, max_len: int = 64, add_bos_eos: bool = True) -> torch.LongTensor:
    s = ''.join([c for c in seq.upper() if c in AA2IDX])
    toks = []
    if add_bos_eos:
        toks.append(IDX_BOS)
    toks += [AA2IDX[c] for c in s]
    if add_bos_eos:
        toks.append(IDX_EOS)
    toks = toks[:max_len]
    pad_len = max_len - len(toks)
    if pad_len > 0:
        toks += [IDX_PAD] * pad_len
    return torch.tensor(toks, dtype=torch.long)

# ========= Dataset =========
class PepTextCSV(Dataset):
    """
    CSV columns expected: id, Sequence/sequence, task, target, notes (extra ok).
    """
    def __init__(self, csv_path: str, max_len: int = 64,
                 text_field_order=("target","notes","task")):
        assert os.path.isfile(csv_path), f"CSV not found: {csv_path}"
        self.rows = []
        with open(csv_path, newline='', encoding='utf-8') as f:
            r = csv.DictReader(f)
            for row in r:
                seq = row.get("Sequence") or row.get("sequence")
                if not seq:
                    continue
                self.rows.append(row)
        self.max_len = max_len
        self.text_field_order = text_field_order

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        seq = row.get("Sequence") or row.get("sequence") or ""
        toks = encode_seq(seq, self.max_len, add_bos_eos=True)
        fields = []
        for k in self.text_field_order:
            v = (row.get(k) or "").strip()
            if v:
                fields.append(f"{k}: {v}")
        if not fields:
            fields = ["target: unspecified"]
        text = " ; ".join(fields)
        return {"id": row.get("id", str(idx)), "tokens": toks, "text": text}

def pep_collate(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    ids = [b["id"] for b in batch]
    toks = torch.stack([b["tokens"] for b in batch], dim=0)  # [B,L]
    texts = [b["text"] for b in batch]
    return {"ids": ids, "tokens": toks, "texts": texts}

# ========= Utils =========
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

# ========= Text side: SciBERT (frozen) + CLIP head + DDPM-Adapter =========
try:
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    AutoTokenizer, AutoModel = None, None
    print("HuggingFace 'transformers' not found. Install with: pip install transformers", file=sys.stderr)

class TextSide(nn.Module):
    def __init__(self, model_name="allenai/scibert_scivocab_uncased",
                 clip_dim=256, freeze_bert=True):
        super().__init__()
        if AutoModel is None:
            raise RuntimeError("transformers package required for SciBERT.")
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False
        h = self.bert.config.hidden_size  # 768
        # CLIP head: 768 -> 256
        self.clip_proj = nn.Linear(h, clip_dim)

        # DDPM-style Adapter: takes z_txt_clip + t_emb -> c_t (256)
        self.t_emb = SinusoidalPosEmb(clip_dim)  # time embedding to 256
        self.adapter = nn.Sequential(
            nn.Linear(clip_dim*2, clip_dim),
            nn.GELU(),
            nn.Linear(clip_dim, clip_dim)
        )

    @torch.no_grad()
    def _raw_cls(self, texts, device):
        enc = self.tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        return self.bert(**enc).last_hidden_state[:, 0]  # [B,768]

    def forward_clip(self, texts: List[str], device) -> torch.Tensor:
        """Return z_txt_clip [B,256] for CLIP (trainable, grads flow)."""
        # We want grads for clip_proj but not for SciBERT (frozen), so we detach raw_cls.
        with torch.no_grad():
            raw = self._raw_cls(texts, device)  # [B,768], no grad
        z = self.clip_proj(raw)                 # [B,256], trainable
        return z

    def forward_c_t(self, z_txt_clip: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Return c_t [B,256] (time-aware text condition for diffusion)."""
        t_e = self.t_emb(t)                  # [B,256]
        x = torch.cat([z_txt_clip, t_e], dim=-1)
        return self.adapter(x)               # [B,256]

# ========= Peptide side: Encoder + trainable projection =========
class PepEncoder(nn.Module):
    """Token encoder for peptide sequences. Causal encoder over tokens."""
    def __init__(self, vocab_size=VOCAB_SIZE, d0=256, layers=4, nhead=8, mlp_ratio=4.0, causal=True, max_len=64):
        super().__init__()
        self.d0 = d0
        self.emb = nn.Embedding(vocab_size, d0)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d0))
        enc_layer = nn.TransformerEncoderLayer(d_model=d0, nhead=nhead,
                                               dim_feedforward=int(d0*mlp_ratio),
                                               batch_first=True, activation="gelu")
        self.causal = causal
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)

    def _causal_mask(self, L: int, device):
        if not self.causal:
            return None
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        # tokens: [B,L]
        B, L = tokens.shape
        x = self.emb(tokens) + self.pos[:, :L, :]  # [B,L,d0]
        mask = self._causal_mask(L, tokens.device)
        # PyTorch Transformer expects float mask as -inf where masked; build attn_mask accordingly
        attn_mask = None
        if mask is not None:
            attn_mask = torch.zeros(L, L, device=tokens.device, dtype=torch.float32)
            attn_mask.masked_fill_(mask, float('-inf'))
        h = self.encoder(x, mask=attn_mask)       # [B,L,d0]
        return h

class PepProj(nn.Module):
    """Per-token trainable projection: d0 -> d (kept trainable even if d0==d)."""
    def __init__(self, d0: int, d: int):
        super().__init__()
        self.ln = nn.LayerNorm(d0)
        self.fc1 = nn.Linear(d0, d)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(d, d)
        self.use_res = (d0 == d)
        nn.init.trunc_normal_(self.fc2.weight, std=1e-3)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, p0: torch.Tensor) -> torch.Tensor:
        # p0: [B,L,d0]
        x = self.ln(p0)
        y = self.fc2(self.act(self.fc1(x)))  # [B,L,d]
        if self.use_res:
            y = y + p0
        return y  # p0_proj

# ========= Diffusion backbone (causal DiT with channel-concat conditioning) =========
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: [B] (int or float)
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=device).float() / max(1, (half - 1)))
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)   # [B,half]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # [B,dim or dim-1]
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb  # [B,dim]

class FiLM(nn.Module):
    def __init__(self, d: int, cond_dim: int):
        super().__init__()
        self.fc = nn.Linear(cond_dim, 2*d)
    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        # x: [B,L,d], c: [B,cond_dim]
        scale, shift = self.fc(c).chunk(2, dim=-1)  # [B,d], [B,d]
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class DiTBlock(nn.Module):
    def __init__(self, d=256, nhead=8, mlp_ratio=4.0, cond_dim=256, causal=True):
        super().__init__()
        self.causal = causal
        self.self_attn = nn.MultiheadAttention(d, nhead, batch_first=True)
        self.ln1 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, int(d*mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(d*mlp_ratio), d)
        )
        self.film = FiLM(d, cond_dim)

    def _attn_mask(self, L, device):
        if not self.causal: return None
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, x: torch.Tensor, c_t: torch.Tensor) -> torch.Tensor:
        # FiLM by c_t
        x = self.film(x, c_t)
        # causal self-attention
        attn_mask = self._attn_mask(x.size(1), x.device)
        x2 = self.ln1(x)
        x2, _ = self.self_attn(x2, x2, x2, attn_mask=attn_mask)
        x = x + x2
        x = x + self.ffn(x)
        return x

class CausalDiT(nn.Module):
    def __init__(self, d=256, layers=6, nhead=8, mlp_ratio=4.0, cond_dim=256, vocab_size=VOCAB_SIZE):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.LayerNorm(d + cond_dim),
            nn.Linear(d + cond_dim, d)
        )
        self.blocks = nn.ModuleList([DiTBlock(d, nhead, mlp_ratio, cond_dim, causal=True) for _ in range(layers)])
        self.out_eps = nn.Linear(d, d)  # predict ε
        self.dec_head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, vocab_size))

    def forward(self, p_t: torch.Tensor, c_t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        p_t: [B,L,d]  (noised peptide features)
        c_t: [B,256]  (time-aware text condition)
        """
        B, L, d = p_t.shape
        C = c_t.unsqueeze(1).expand(B, L, c_t.size(-1))  # [B,L,256]
        x = torch.cat([p_t, C], dim=-1)                  # [B,L,d+256]
        h = self.fuse(x)                                 # [B,L,d]
        for blk in self.blocks:
            h = blk(h, c_t)                              # FiLM with c_t
        eps_pred = self.out_eps(h)                       # [B,L,d]
        logits   = self.dec_head(h)                      # [B,L,V]
        return eps_pred, logits

# ========= Losses & helpers =========
def nonpad_mask(tokens: torch.Tensor) -> torch.Tensor:
    return (tokens != IDX_PAD).float()  # [B,L]

def aa_mask(tokens: torch.Tensor) -> torch.Tensor:
    return ((tokens >= 0) & (tokens < 20)).float()

def masked_mse_eps(pred: torch.Tensor, target: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    # pred/target: [B,L,d]
    mask = nonpad_mask(tokens)  # [B,L]
    diff = (pred - target).pow(2).mean(dim=-1)  # [B,L]
    return (diff * mask).sum() / (mask.sum() + 1e-8)

def ce_aa_plus_eos(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Cross-entropy computed ONLY on AA (0..19) and EOS (20). BOS and PAD are ignored.
    """
    B, L, V = logits.shape
    ce = F.cross_entropy(logits.view(B*L, V), targets.view(B*L), reduction='none')  # [B*L]
    ce = ce.view(B, L)
    valid = ((targets >= 0) & (targets < 20)) | (targets == IDX_EOS)
    valid = valid.float()
    return (ce * valid).sum() / (valid.sum() + 1e-8)

def aa_mean_pool(feats: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    """
    Mean over AA positions only (exclude BOS/EOS/PAD).
    feats: [B,L,D]
    """
    mask = aa_mask(tokens)  # [B,L]
    num = (feats * mask.unsqueeze(-1)).sum(dim=1)        # [B,D]
    den = mask.sum(dim=1, keepdim=True) + 1e-8
    return num / den

def info_nce(z_text: torch.Tensor, z_pep: torch.Tensor, tau=0.07) -> torch.Tensor:
    zt = F.normalize(z_text, dim=-1)
    zp = F.normalize(z_pep,  dim=-1)
    logits = (zt @ zp.t()) / tau  # [B,B]
    labels = torch.arange(zt.size(0), device=zt.device)
    return F.cross_entropy(logits, labels)

# ========= Betas schedule =========
def linear_beta_schedule(T=1000, start=1e-4, end=2e-2, device="cpu"):
    betas = torch.linspace(start, end, T, device=device)
    alphas = 1.0 - betas
    alphas_cum = torch.cumprod(alphas, dim=0)
    return betas, alphas, alphas_cum

# ========= Config =========
@dataclass
class Config:
    train_csv: str = "train.csv"
    val_csv: str = "val.csv"
    out_dir: str = "litepep_logs"
    device: str = "cuda"
    seed: int = 42
    # model dims
    max_len: int = 50
    d0: int = 256
    d:  int = 256
    layers_pep: int = 4
    layers_dit: int = 6
    nhead: int = 8
    mlp_ratio: float = 4.0
    # opt
    batch_size: int = 32
    epochs: int = 20
    lr: float = 2e-4
    weight_decay: float = 0.01
    # loss weights
    w_diff: float = 1.0
    w_ce: float = 2.0
    w_align: float = 1.0
    w_clip: float = 1.0
    clip_tau: float = 0.07
    log_every: int = 100

# ========= Training/Eval =========
def train_one_epoch(textside: TextSide, pepenc: PepEncoder, pepproj: PepProj, dit: CausalDiT,
                    loader, optimizer, cfg: Config, device, alphas_cum):
    textside.train()   # Only CLIP head + Adapter have grads; BERT stays frozen.
    pepenc.train()
    pepproj.train()
    dit.train()

    totals = {"diff":0.0,"ce":0.0,"align":0.0,"clip":0.0,"n":0}
    step = 0
    for batch in loader:
        step += 1
        tokens = batch["tokens"].to(device)  # [B,L]
        texts  = batch["texts"]
        B, L = tokens.shape

        # ----- Stage-2: peptide -----
        p0       = pepenc(tokens)                       # [B,L,d0] clean
        p0_proj  = pepproj(p0)                          # [B,L,d]  for diffusion & align
        p_pool   = aa_mean_pool(p0, tokens)             # [B,d0]   for CLIP (clean)

        # ----- Stage-1: text -----
        z_txt    = textside.forward_clip(texts, device) # [B,256]  CLIP head output
        # time/sample
        t = torch.randint(0, 1000, (B,), device=device).long()
        c_t      = textside.forward_c_t(z_txt, t)       # [B,256]

        # ----- Diffusion noising (inside) -----
        eps = torch.randn_like(p0_proj)                 # [B,L,d]
        at  = alphas_cum[t].view(B,1,1)                 # [B,1,1]
        p_t = (at.sqrt() * p0_proj) + ((1.0 - at).sqrt() * eps)  # [B,L,d]

        # ----- Diffusion backbone -----
        eps_pred, logits = dit(p_t, c_t)                # [B,L,d], [B,L,V]

        # ----- Losses -----
        L_diff   = masked_mse_eps(eps_pred, eps, tokens)               # ε-MSE
        L_ce     = ce_aa_plus_eos(logits, tokens)                      # CE on AA+EOS only
        L_align  = F.mse_loss(aa_mean_pool(p_t, tokens), c_t)          # per-t alignment
        L_clip   = info_nce(z_txt, p_pool, tau=cfg.clip_tau)           # CLIP on clean p0

        loss = cfg.w_diff*L_diff + cfg.w_ce*L_ce + cfg.w_align*L_align + cfg.w_clip*L_clip

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(pepenc.parameters()) + list(pepproj.parameters()) + list(dit.parameters()) + list(textside.adapter.parameters()) + list(textside.clip_proj.parameters()), 1.0)
        optimizer.step()

        totals["diff"]  += L_diff.item()*B
        totals["ce"]    += L_ce.item()*B
        totals["align"] += L_align.item()*B
        totals["clip"]  += L_clip.item()*B
        totals["n"]     += B

        if (step % cfg.log_every) == 0:
            print(f"  step {step:5d}: L_diff={L_diff.item():.4f}  L_ce={L_ce.item():.4f}  L_align={L_align.item():.4f}  L_clip={L_clip.item():.4f}")

    n = totals["n"]
    return {k: totals[k]/n for k in ["diff","ce","align","clip"]}

@torch.no_grad()
def evaluate(textside: TextSide, pepenc: PepEncoder, pepproj: PepProj, dit: CausalDiT,
             loader, cfg: Config, device, alphas_cum):
    textside.eval(); pepenc.eval(); pepproj.eval(); dit.eval()
    totals = {"diff":0.0,"ce":0.0,"align":0.0,"clip":0.0,"acc":0.0,"n":0}
    for batch in loader:
        tokens = batch["tokens"].to(device)
        texts  = batch["texts"]
        B, L = tokens.shape

        p0      = pepenc(tokens)
        p0_proj = pepproj(p0)
        p_pool  = aa_mean_pool(p0, tokens)

        z_txt   = textside.forward_clip(texts, device)
        t       = torch.randint(0, 1000, (B,), device=device).long()
        c_t     = textside.forward_c_t(z_txt, t)

        eps = torch.randn_like(p0_proj)
        at  = alphas_cum[t].view(B,1,1)
        p_t = (at.sqrt() * p0_proj) + ((1.0 - at).sqrt() * eps)

        eps_pred, logits = dit(p_t, c_t)

        L_diff  = masked_mse_eps(eps_pred, eps, tokens)
        L_ce    = ce_aa_plus_eos(logits, tokens)
        L_align = F.mse_loss(aa_mean_pool(p_t, tokens), c_t)
        L_clip  = info_nce(z_txt, p_pool, tau=cfg.clip_tau)

        # token accuracy over non-PAD positions
        pred_tok = logits.argmax(-1)
        nonpad = nonpad_mask(tokens)
        correct = ((pred_tok == tokens).float() * nonpad).sum()
        total_tok = nonpad.sum()
        acc = (correct / (total_tok + 1e-8)).item()

        totals["diff"]  += L_diff.item()*B
        totals["ce"]    += L_ce.item()*B
        totals["align"] += L_align.item()*B
        totals["clip"]  += L_clip.item()*B
        totals["acc"]   += acc*B
        totals["n"]     += B

    n = totals["n"]
    return {k: totals[k]/n for k in ["diff","ce","align","clip","acc"]}

def save_ckpt(textside, pepenc, pepproj, dit, cfg: Config, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "textside_adapter": textside.adapter.state_dict(),
        "textside_clip": textside.clip_proj.state_dict(),
        "pepenc": pepenc.state_dict(),
        "pepproj": pepproj.state_dict(),
        "dit": dit.state_dict(),
        "cfg": asdict(cfg)
    }, path)

def main(cfg: Config):
    os.makedirs(cfg.out_dir, exist_ok=True)
    with open(os.path.join(cfg.out_dir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)

    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    # Data
    train_set = PepTextCSV(cfg.train_csv, max_len=cfg.max_len)
    val_set   = PepTextCSV(cfg.val_csv,   max_len=cfg.max_len)
    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, collate_fn=pep_collate)
    val_loader   = DataLoader(val_set,   batch_size=cfg.batch_size, shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=pep_collate)

    # Models
    textside = TextSide(freeze_bert=True).to(device)             # SciBERT frozen
    pepenc   = PepEncoder(d0=cfg.d0, layers=cfg.layers_pep,
                          nhead=cfg.nhead, mlp_ratio=cfg.mlp_ratio,
                          causal=True, max_len=cfg.max_len).to(device)
    pepproj  = PepProj(d0=cfg.d0, d=cfg.d).to(device)
    dit      = CausalDiT(d=cfg.d, layers=cfg.layers_dit, nhead=cfg.nhead,
                         mlp_ratio=cfg.mlp_ratio, cond_dim=256, vocab_size=VOCAB_SIZE).to(device)

    # Opt
    params = list(textside.adapter.parameters()) + list(textside.clip_proj.parameters()) + \
             list(pepenc.parameters()) + list(pepproj.parameters()) + list(dit.parameters())
    optimizer = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    # Schedule
    betas, alphas, alphas_cum = linear_beta_schedule(T=1000, device=device)

    # CSV log
    log_path = os.path.join(cfg.out_dir, "train_log.csv")
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["epoch","split","L_diff","L_ce","L_align","L_clip","tok_acc"])

    best = float("inf")
    for ep in range(1, cfg.epochs+1):
        t0 = time.time()
        tr = train_one_epoch(textside, pepenc, pepproj, dit, train_loader, optimizer, cfg, device, alphas_cum)
        va = evaluate(textside, pepenc, pepproj, dit, val_loader, cfg, device, alphas_cum)
        dt = time.time() - t0

        with open(log_path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow([ep,"train", f"{tr['diff']:.6f}", f"{tr['ce']:.6f}", f"{tr['align']:.6f}", f"{tr['clip']:.6f}", ""])
            w.writerow([ep,"val",   f"{va['diff']:.6f}", f"{va['ce']:.6f}", f"{va['align']:.6f}", f"{va['clip']:.6f}", f"{va['acc']:.4f}"])

        print(f"Epoch {ep:03d} ({dt:.1f}s)  "
              f"Train: L_diff={tr['diff']:.4f}  L_ce={tr['ce']:.4f}  L_align={tr['align']:.4f}  L_clip={tr['clip']:.4f}  |  "
              f"Val: L_diff={va['diff']:.4f}  L_ce={va['ce']:.4f}  L_align={va['align']:.4f}  L_clip={va['clip']:.4f}  tok_acc={va['acc']:.3f}")

        val_score = va["ce"] + va["diff"]
        if val_score < best:
            best = val_score
            save_ckpt(textside, pepenc, pepproj, dit, cfg, os.path.join(cfg.out_dir, "best.pt"))
        save_ckpt(textside, pepenc, pepproj, dit, cfg, os.path.join(cfg.out_dir, "last.pt"))




In [3]:
if __name__ == "__main__":
    cfg = Config(
        train_csv="/content/drive/MyDrive/TGPepGM/final_datasets/train_acp.csv",
        val_csv="/content/drive/MyDrive/TGPepGM/final_datasets/test_acp.csv",
        out_dir="/content/drive/MyDrive/TGPepGM/litepep_logs_100epoch",
        device="cuda" if torch.cuda.is_available() else "cpu",
        seed=42,
        max_len=50,
        d0=256,
        d=256,
        layers_pep=4,
        layers_dit=6,
        nhead=8,
        mlp_ratio=4.0,
        batch_size=128,
        epochs=100,
        lr=2e-3,
        weight_decay=0.01,
        w_diff=1.0,
        w_ce=1.0,         # CE weight per your request
        w_align=1.0,
        w_clip=1.0,
        clip_tau=0.2,
        log_every=100
    )
    main(cfg)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Epoch 001 (20.9s)  Train: L_diff=1.0658  L_ce=1.0015  L_align=0.1906  L_clip=4.8571  |  Val: L_diff=0.6356  L_ce=0.5330  L_align=0.2185  L_clip=4.8496  tok_acc=0.798
Epoch 002 (19.9s)  Train: L_diff=0.4547  L_ce=0.5485  L_align=0.1787  L_clip=4.8530  |  Val: L_diff=0.4052  L_ce=0.4672  L_align=0.2181  L_clip=4.8499  tok_acc=0.812
Epoch 003 (21.7s)  Train: L_diff=0.3247  L_ce=0.4890  L_align=0.1766  L_clip=4.8537  |  Val: L_diff=0.3634  L_ce=0.4562  L_align=0.2195  L_clip=4.8495  tok_acc=0.817
Epoch 004 (24.3s)  Train: L_diff=0.2865  L_ce=0.4514  L_align=0.1763  L_clip=4.8523  |  Val: L_diff=0.2751  L_ce=0.4999  L_align=0.1987  L_clip=4.8498  tok_acc=0.802
Epoch 005 (25.8s)  Train: L_diff=0.2599  L_ce=0.4552  L_align=0.1776  L_clip=4.8530  |  Val: L_diff=0.2753  L_ce=0.5598  L_align=0.2459  L_clip=4.8495  tok_acc=0.798
Epoch 006 (23.4s)  Train: L_diff=0.2405  L_ce=0.5135  L_align=0.1904  L_clip=4.8531  |  Val: L_diff=0.2589  L_ce=0.4279  L_align=0.1857  L_clip=4.8498  tok_acc=0.823
Epoc

In [4]:
# litepep_generate.py

import os, math, torch
import torch.nn.functional as F

# --- Reuse constants from training ---
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"
IDX_EOS, IDX_PAD, IDX_BOS = 20, 21, 22
VOCAB_SIZE = 23
IDX2AA = {i: a for i, a in enumerate(AA_ALPHABET)}

# --------- helpers (match training schedule & masks) ----------
def linear_beta_schedule(T=1000, start=1e-4, end=2e-2, device="cpu"):
    betas = torch.linspace(start, end, T, device=device)
    alphas = 1.0 - betas
    alphas_cum = torch.cumprod(alphas, dim=0)
    return betas, alphas, alphas_cum

def mask_logits_for_decoding(logits, min_len=3):
    """
    Forbid BOS & PAD everywhere; forbid EOS for first 'min_len' positions.
    logits: [B, L, V]
    """
    B, L, V = logits.shape
    logits = logits.clone()

    # forbid BOS / PAD everywhere
    logits[..., IDX_BOS] = -float("inf")
    logits[..., IDX_PAD] = -float("inf")

    # forbid EOS before min_len using a clean slice on the length axis
    if min_len > 0:
        m = min_len if isinstance(min_len, int) else int(min_len)
        m = max(0, min(m, L))  # clamp
        if m > 0:
            logits[:, :m, IDX_EOS] = -float("inf")

    return logits

def tokens_to_strings(tokens_2d):
    """
    tokens_2d: [B,L] ints; decode AAs until EOS, skip BOS/PAD
    """
    seqs = []
    for row in tokens_2d.tolist():
        out = []
        for t in row:
            if t == IDX_EOS:
                break
            if 0 <= t < 20:
                out.append(IDX2AA[t])
            # skip BOS/PAD
        seqs.append("".join(out))
    return seqs

# --------- SAMPLER (DDIM, eta=0) ----------
@torch.no_grad()
def ddim_sample(textside, pepenc, pepproj, dit,
                texts,
                max_len=64, d=256,
                steps=50, T=1000,
                device="cuda",
                min_len_before_eos=3,
                temperature=1.0):
    """
    Returns: list of peptide strings (len = len(texts))
    """
    device = torch.device(device)
    B = len(texts)

    # Schedules
    _, _, alphas_cum = linear_beta_schedule(T=T, device=device)  # [T]
    # choose integer t indices for DDIM steps, descending
    ts = torch.linspace(T-1, 0, steps, device=device).long()     # [S]

    # Text → z_txt_clip (CLIP) → c_t per step
    z_txt_clip = textside.forward_clip(texts, device=device)     # [B,256]

    # Initialize latent at t=T: p_T ~ N(0, I)
    p_t = torch.randn(B, max_len, d, device=device)              # [B,L,d]

    # Reverse steps
    for i in range(steps):
        t = ts[i]                                                # scalar int
        a_t = alphas_cum[t]                                      # []
        # time-conditioned text condition
        c_t = textside.forward_c_t(z_txt_clip, t.expand(B))      # [B,256]
        # Model predicts eps at current (p_t, t, c_t)
        eps_pred, _ = dit(p_t, c_t)                              # [B,L,d], logits unused mid-steps
        # DDIM update (eta=0): x0_pred and deterministic step to t_prev
        a_t_sqrt = a_t.sqrt().view(1,1,1)
        one_minus_a_t_sqrt = (1.0 - a_t).sqrt().view(1,1,1)
        x0_pred = (p_t - one_minus_a_t_sqrt * eps_pred) / (a_t_sqrt + 1e-8)

        if i == steps - 1:
            p_t = x0_pred                                        # at t=0, done
        else:
            t_prev = ts[i+1]
            a_prev = alphas_cum[t_prev]
            a_prev_sqrt = a_prev.sqrt().view(1,1,1)
            one_minus_a_prev_sqrt = (1.0 - a_prev).sqrt().view(1,1,1)
            # DDIM (eta=0): x_{t-1} = sqrt(a_prev)*x0_pred + sqrt(1-a_prev)*eps_pred
            p_t = a_prev_sqrt * x0_pred + one_minus_a_prev_sqrt * eps_pred

    # Decode tokens at t=0 (use DiT once more to get logits)
    c_0 = textside.forward_c_t(z_txt_clip, torch.zeros(B, device=device, dtype=torch.long))
    _, logits = dit(p_t, c_0)                                    # [B,L,V]
    if temperature != 1.0:
        logits = logits / max(1e-8, float(temperature))

    logits = mask_logits_for_decoding(logits, min_len=min_len_before_eos)
    tokens = logits.argmax(-1)                                   # [B,L]
    return tokens_to_strings(tokens)

# --------- LOADER ----------
def load_models_from_ckpt(ckpt_path, cfg, device="cuda"):
    """
    Rebuilds model objects exactly like training and loads weights.
    - 'cfg' must carry the same fields used at train time (d0, d, layers, etc.).
    """
    device = torch.device(device)
    # Recreate modules (must match training)
    textside = TextSide(freeze_bert=True).to(device)
    pepenc   = PepEncoder(d0=cfg.d0, layers=cfg.layers_pep,
                          nhead=cfg.nhead, mlp_ratio=cfg.mlp_ratio,
                          causal=True, max_len=cfg.max_len).to(device)
    pepproj  = PepProj(d0=cfg.d0, d=cfg.d).to(device)
    dit      = CausalDiT(d=cfg.d, layers=cfg.layers_dit, nhead=cfg.nhead,
                         mlp_ratio=cfg.mlp_ratio, cond_dim=256, vocab_size=VOCAB_SIZE).to(device)

    sd = torch.load(ckpt_path, map_location=device)
    textside.adapter.load_state_dict(sd["textside_adapter"])
    textside.clip_proj.load_state_dict(sd["textside_clip"])
    pepenc.load_state_dict(sd["pepenc"])
    pepproj.load_state_dict(sd["pepproj"])
    dit.load_state_dict(sd["dit"])
    for m in (textside, pepenc, pepproj, dit):
        m.eval()
    return textside, pepenc, pepproj, dit

# --------- ONE-CALL API ----------
@torch.no_grad()
def generate(texts, ckpt_path, cfg, steps=50, device="cuda",
             min_len_before_eos=3, temperature=1.0):
    """
    texts: list[str] of prompts/conditions
    Returns: list[str] peptides (same length as texts)
    """
    textside, pepenc, pepproj, dit = load_models_from_ckpt(ckpt_path, cfg, device=device)

    # Note: PepEncoder is only used in training; at generation we start from noise.
    # We rebuild sizes from cfg to set L and d for the sampler.
    peptides = ddim_sample(
        textside=textside, pepenc=pepenc, pepproj=pepproj, dit=dit,
        texts=texts, max_len=cfg.max_len, d=cfg.d,
        steps=steps, T=1000, device=device,
        min_len_before_eos=min_len_before_eos, temperature=temperature
    )
    return peptides

# ------------------ Example usage ------------------
if __name__ == "__main__":
    # Reuse the same Config dataclass from training (import it or replicate fields here)
    from types import SimpleNamespace
    cfg = SimpleNamespace(
        max_len=50, d0=256, d=256, layers_pep=4, layers_dit=6, nhead=8, mlp_ratio=4.0
    )
    ckpt = "/content/drive/MyDrive/TGPepGM/litepep_logs_30epoch/best.pt"
    texts = [
        "task: ACP ; target: MCF-7 ; notes: short, non-hemolytic",
    ]
    out = generate(texts, ckpt, cfg, steps=50, device="cuda", min_len_before_eos=4)
    for i, s in enumerate(out):
        print(f"[{i}] {s}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0] LRGGFCHCHTGPRFSQFRNRDNKNKNEKSRPRAKVFALLKLIAALSYGCA


In [5]:
# === Generate 1000 peptides and save to FASTA ===
import os
from types import SimpleNamespace

# Reuse your existing generate(), load_models_from_ckpt(), and all classes imported above.

def write_fasta(seqs, path, start_idx=1, header_prefix="pep"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for i, s in enumerate(seqs, start_idx):
            f.write(f">{header_prefix}_{i}\n{s}\n")

def batched_generate(text_prompt, n_total, batch_size, ckpt_path, cfg,
                     steps=50, device="cuda", min_len_before_eos=4, temperature=1.0):
    """Generate n_total sequences in batches with the same text condition."""
    all_peps = []
    remaining = n_total
    while remaining > 0:
        bsz = min(batch_size, remaining)
        texts = [text_prompt] * bsz
        peps = generate(
            texts=texts,
            ckpt_path=ckpt_path,
            cfg=cfg,
            steps=steps,
            device=device,
            min_len_before_eos=min_len_before_eos,
            temperature=temperature
        )
        all_peps.extend(peps)
        remaining -= bsz
    return all_peps

if __name__ == "__main__":
    # Must match the training config (dims/layers)
    cfg = SimpleNamespace(
        max_len=50, d0=256, d=256, layers_pep=4, layers_dit=6, nhead=8, mlp_ratio=4.0
    )

    ckpt_path = "/content/drive/MyDrive/TGPepGM/litepep_logs_30epoch/best.pt"   # <-- adjust if needed
    out_fasta = "/content/drive/MyDrive/TGPepGM/litepep_logs_30epoch/generated_1000.fasta"

    # Choose your condition text (you can swap or loop over multiple prompts if desired)
    text_prompt = "This is a peptide: cancer type:Breast Cancer cell line:MCF-7 tissue source:Breast activity:Tetrazolium based assay topology:Linear chirality:l length:20 n term mod:Free cterm mod:Free"

    # Generate 1000 peptides in batches of 64 (you can change batch_size)
    peptides = batched_generate(
        text_prompt=text_prompt,
        n_total=1000,
        batch_size=64,
        ckpt_path=ckpt_path,
        cfg=cfg,
        steps=60,                      # a bit higher for quality
        device="cuda",
        min_len_before_eos=4,          # avoid ultra-short sequences
        temperature=1.0                # set >1.0 for more diversity
    )

    # Save to FASTA
    write_fasta(peptides, out_fasta, start_idx=1, header_prefix="litepep")
    print(f"Saved {len(peptides)} peptides to {out_fasta}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved 1000 peptides to /content/drive/MyDrive/TGPepGM/litepep_logs_30epoch/generated_1000.fasta


## Evaluation (AM-Score)

The 1,000 generated samples are evaluated with AMP prediction tools. The notebook uses amPEP and IPPF-FE from their GitHub implementations and CAMP from the CAMP portal. For each available model, the AM-Score is calculated as the fraction of generated sequences predicted as antimicrobial peptides.

This notebook is an experimental lightweight extension and the evaluation was not completed as a full research study.


### Get amPEP predictions directly

The pretrained amPEP model is loaded from its GitHub implementation and run on the generated FASTA file. Prediction probabilities are saved and used in the final AM-Score calculation.


In [ ]:
#  Install
%cd  /content
!pip -q install git+https://github.com/tlawrence3/amPEPpy.git biopython pandas

import os, re, json, pathlib, pandas as pd
from pathlib import Path

# INPUT (FASTA or JSON/JSONL)
IN_PATH = "/content/drive/MyDrive/litepep_logs/generated_1000.fasta"   # change this

def to_fasta(in_path, out_path="/content/axpep_input.fasta"):
    p = Path(in_path)
    def norm(s):
        s = re.sub(r"\s+","", s).upper()
        s = re.sub(r"[UZOBJ]","X", s)
        return re.sub(r"[^A-Z]","", s)
    if p.suffix.lower() in [".json",".jsonl"]:
        def iter_json():
            if p.suffix.lower()==".jsonl":
                for i,l in enumerate(open(p,encoding="utf-8"),1):
                    if l.strip():
                        o=json.loads(l)
                        yield f"pep_{i:06d}", next((o.get(k,"") for k in ["sequence","seq","peptide","trg","target","y"]), "")
            else:
                data=json.load(open(p,encoding="utf-8"))
                if isinstance(data,dict):
                    it=list(data.items())
                elif isinstance(data,list):
                    it=[(f"pep_{i:06d}", (x.get("sequence") or x.get("seq") or x.get("peptide") or x.get("trg") or x.get("target") or x.get("y",""))) for i,x in enumerate(data,1)]
                else:
                    it=[]
                for k,v in it: yield str(k), v
        with open(out_path,"w") as w:
            seen=set()
            for hid,seq in iter_json():
                s=norm(str(seq))
                if 4<=len(s)<=1000 and s not in seen:
                    seen.add(s); w.write(f">{hid}\n{s}\n")
        return out_path
    return in_path

FASTA = to_fasta(IN_PATH)

# ====== 2) Locate or fetch the amPEP model file ======
MODEL_PATH = None
try:
    import amPEPpy
    pkg = Path(amPEPpy.__file__).parent
    cands = list(pkg.rglob("*.model"))
    if cands:
        MODEL_PATH = str(cands[0])
except Exception:
    pass

if MODEL_PATH is None:
    # fallback: clone repo to get pretrained model

    !rm -rf /content/amPEPpy_repo
    !git clone -q https://github.com/tlawrence3/amPEPpy.git /content/amPEPpy_repo
    MODEL_PATH = "/content/amPEPpy_repo/pretrained_models/amPEP.model"
    assert os.path.exists(MODEL_PATH), "Could not find amPEP.model after cloning."

print("Using model:", MODEL_PATH)
print("Using FASTA:", FASTA)

# Run amPEP and compute AMscore
!ampep predict -m "$MODEL_PATH" -i "$FASTA" -o "/content/ampep_scores.tsv"

df = pd.read_csv("/content/ampep_scores.tsv", sep="\t")
prob_col = next(c for c in df.columns if any(k in c.lower() for k in ["prob","score","amp"]))
ams = (df[prob_col] >= 0.5).mean()
print(f"\nAMscore (amPEP, thr=0.5): {ams:.3f}  | N={len(df)}  | prob_col='{prob_col}'")
display(df.head(3))

/content
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.1 MB/s eta 0:00:00
Using model: /content/amPEPpy_repo/pretrained_models/amPEP.model
Using FASTA: /content/drive/MyDrive/litepep_logs/generated_1000.fasta
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persis

,probability_nonAMP,probability_AMP,predicted,seq_id
0,0.240625,0.759375,AMP,litepep_1
1,0.206250,0.793750,AMP,litepep_2
2,0.312500,0.687500,AMP,litepep_3


### Get IPPF-FE predictions directly

The IPPF-FE repository is cloned and its prediction pipeline is run on the generated FASTA samples. This part was exploratory and the saved notebook also keeps the dependency/model-loading issues that appeared during the run.


In [ ]:
%cd  /content
!git clone https://github.com/HanselYu/IPPF-FE
%cd  /content/IPPF-FE

/content
Cloning into 'IPPF-FE'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 106 (delta 17), reused 26 (delta 11), pack-reused 70 (from 1)
Receiving objects: 100% (106/106), 4.61 MiB | 10.21 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/IPPF-FE


In [ ]:
!pip install -r requirment.txt
!pip -q install --upgrade pip
!pip -q install "torch==2.3.1" "torchvision==0.18.1" --index-url https://download.pytorch.org/whl/cu121

!pip -q install transformers sentencepiece pandas numpy scikit-learn lightgbm biopython
# 1) Make sure deps are fine (skip the repo's outdated requirment.txt)
!pip -q install --upgrade transformers safetensors sentencepiece pandas numpy scikit-learn lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.1/484.1 kB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Ignored the following versions that require a different python version: 0.7 Requires-Python >=3.6, <3.7; 0.8 Requires-Python >=3.6, <3.7
ERROR: Could not find a version that satisfies the requirement dataclasses==0.8 (from versions: 0.1, 0.2, 0.3, 0.4, 0.5, 0.6)
ERROR: No matching distribution found for dataclasses==0.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.8.0+cu126 requires torch==2.8.0, but you have torch 2.3.1+cu121 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 req

In [ ]:
%cd /content/IPPF-FE
!python Pantibacterial.py Train -

/content/IPPF-FE
2025-10-29 14:21:22.347902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761747682.367957   32771 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761747682.374120   32771 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761747682.389227   32771 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761747682.389258   32771 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761747682.389262   32771 computation_placer.cc:177] compu

In [ ]:
from pathlib import Path
import re

p = Path("/content/IPPF-FE/Pantibacterial.py")
s = p.read_text()

s = s.replace('"prot_t5_xl_uniref50"', '"Rostlab/prot_t5_xl_uniref50"')

s = s.replace(
    'T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")',
    'T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50", use_safetensors=True, torch_dtype=torch.float16)'
)

s = s.replace(
    "torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')",
    "torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
)

if "import numpy as np" not in s:
    s = s.replace("import sys", "import sys\nimport numpy as np")
if "import pandas as pd" not in s:
    s = s.replace("import sys", "import sys\nimport pandas as pd")

p.write_text(s)
print("Patched Pantibacterial.py")
print("Leftover bad IDs:", re.findall(r"prot_t5_xl_uniref50", s))


In [ ]:
!python Pantibacterial.py Predict /content/TG-CDDPM/sample/test.fasta

2025-09-20 06:30:50.475865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758349850.497014   27381 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758349850.503609   27381 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758349850.520005   27381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758349850.520033   27381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758349850.520036   27381 computation_placer.cc:177] computation placer alr

## Final results

The available evaluation outputs are combined at the end of the notebook. These results are preliminary and belong to an unfinished lightweight-model experiment, so they are kept here as a record of the work rather than a final benchmark.


In [ ]:
import pandas  as pd
# ippf_fe_output = pd.read_excel("/content/IPPF-FE/Pre_label.xlsx")
Camp_output = pd.read_csv("/content/CAMPdownload_2025-10-29 19-45-29.txt",sep='\t')
Axpep = pd.read_csv("/content/ampep_scores.tsv",sep='\t')

In [ ]:
final_results = {'CAMP': [(pd.get_dummies(Camp_output['Class'], dtype=int)['AMP']).mean()],
#  'IPPF-FE': [(ippf_fe_output['Pre_label']).mean()],
 'AxPEP': [(pd.get_dummies(Axpep['predicted'], dtype=int)['AMP']).mean()]
 }
final_results = pd.DataFrame(final_results,index=['AM-Score for 1000 samples'])
final_results['Total'] = final_results.mean(axis=1)

In [ ]:
display(final_results.T)

,AM-Score for 1000 samples
CAMP,0.2130
AxPEP,0.9020
Total,0.5575
